# 🛡️ AEGIS — Live Demo Application
### Upload a CCTV clip → SafetyFormer triage → anonymized evidence → **local Gemma 4** verification → categorized incident decision

This notebook launches a clean web app (Gradio) whose backend is the complete AEGIS pipeline. Run it in an **interactive Kaggle session** (GPU T4 ×2, Internet ON) and it prints a **public shareable URL** judges can open in any browser.

**Setup (2 minutes):**
1. *Add Input* → attach the **UCF-Crime dataset** (`odins0n/ucf-crime-dataset`) — used for sample clips and the quick-train fallback.
2. *(Recommended)* Also attach the **output of the main training notebook** (`aegis-gemma4-public-safety`) — the app then loads the trained SafetyFormer + calibration instantly instead of quick-training one (~12 min).
3. Settings: **Accelerator = GPU T4 ×2**, **Internet = ON**.
4. **Run All.** The last cell prints the app URL.

In [1]:
import importlib, subprocess, sys, warnings
warnings.filterwarnings("ignore")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
                       "-U", "transformers>=5.0", "accelerate", "bitsandbytes", "timm",
                       "gradio>=4.0", "cryptography"], timeout=900)

import os, io, re, json, math, time, random, shutil, hashlib
from pathlib import Path
from types import SimpleNamespace
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import gradio as gr

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
WORK = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path("./demo_out")
WORK.mkdir(exist_ok=True)
import transformers
print(f"torch {torch.__version__} | transformers {transformers.__version__} | "
      f"gradio {gr.__version__} | device {DEVICE}")
if DEVICE.type != "cuda":
    print("WARNING: no GPU — Gemma 4 will not load; enable a GPU session for the demo.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 6.0 MB/s eta 0:00:00
torch 2.10.0+cu128 | transformers 5.13.1 | gradio 6.20.0 | device cuda:0


In [2]:
# ---- locate trained artifacts (same session, or attached training-notebook output)
def find_artifacts():
    roots = [WORK]
    if os.path.isdir("/kaggle/input"):
        for d in sorted(Path("/kaggle/input").iterdir()):
            roots += [d] + [s for s in d.iterdir() if s.is_dir()][:8]
    for r in roots:
        aj, ck = r / "demo_artifacts.json", r / "safetyformer_best.pt"
        if aj.exists() and ck.exists():
            return aj, ck
    return None, None

ART_PATH, CKPT_PATH = find_artifacts()
if ART_PATH:
    ART = json.loads(ART_PATH.read_text())
    print(f"Loaded trained artifacts from {ART_PATH.parent}")
    print(json.dumps({k: v for k, v in ART.items() if k != "anomaly_classes"}, indent=2))
else:
    ART = None
    print("No trained artifacts found -> the quick-train fallback cell below will build a "
          "compact model from the attached dataset (~12 min). For the full-quality model, "
          "attach the main training notebook's output.")

No trained artifacts found -> the quick-train fallback cell below will build a compact model from the attached dataset (~12 min). For the full-quality model, attach the main training notebook's output.


In [3]:
# ================= core AEGIS components (identical logic to the training notebook) =====
import timm

FRAME_SUFFIX_RE = re.compile(r"^(.+?)[_\-. ]?(\d+)$")
NORMAL_KEYWORDS = ("normal", "safe", "benign", "neutral", "no_event", "negative")
XD_VIOLENCE_LABELS = {"B1": "Fighting", "B2": "Shooting", "B4": "Riot",
                      "B5": "Abuse", "B6": "CarAccident", "G": "Explosion"}
XD_LABEL_RE = re.compile(r"label[_\-]([A-Za-z0-9\-]+)", re.I)

def parse_xd_label(stem):
    m = XD_LABEL_RE.search(stem)
    if not m:
        return None
    toks = [t.upper() for t in m.group(1).split("-") if t]
    cats = [XD_VIOLENCE_LABELS[t] for t in toks if t in XD_VIOLENCE_LABELS]
    return cats[0] if cats else ("Normal" if "A" in toks else None)

def build_backbone(model_name):
    m = timm.create_model(model_name, pretrained=True, num_classes=0)
    dc = timm.data.resolve_model_data_config(m)
    spec = dict(name=model_name, size=dc["input_size"][-1], mean=dc["mean"], std=dc["std"])
    m = m.eval().to(DEVICE)
    for p in m.parameters():
        p.requires_grad_(False)
    return m, m.num_features, spec

def preprocess_frames(frames, spec):
    S = spec["size"]
    mean = torch.tensor(spec["mean"]).view(1, 3, 1, 1)
    std = torch.tensor(spec["std"]).view(1, 3, 1, 1)
    x = torch.from_numpy(np.stack([cv2.resize(f, (S, S), interpolation=cv2.INTER_CUBIC)
                                   for f in frames])).permute(0, 3, 1, 2).float().div_(255.0)
    return (x - mean) / std

@torch.no_grad()
def embed_frames(frames, backbone, spec, batch=256):
    out = []
    for s in range(0, len(frames), batch):
        x = preprocess_frames(frames[s:s + batch], spec).to(DEVICE)
        with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
            out.append(backbone(x).half().cpu())
    return torch.cat(out).numpy()

class SafetyFormer(nn.Module):
    def __init__(self, d_in, cfg, n_subtypes):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, cfg.d_model))
        self.pos = nn.Parameter(torch.zeros(1, cfg.window_len, cfg.d_model))
        layer = nn.TransformerEncoderLayer(cfg.d_model, cfg.n_heads, 4 * cfg.d_model,
                                           cfg.dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, cfg.n_layers)
        self.query = nn.Parameter(torch.randn(1, 1, cfg.d_model) * 0.02)
        self.attn_pool = nn.MultiheadAttention(cfg.d_model, cfg.n_heads,
                                               dropout=cfg.dropout, batch_first=True)
        self.norm = nn.LayerNorm(cfg.d_model)
        self.score_head = nn.Linear(cfg.d_model, 1)
        self.subtype_head = nn.Linear(cfg.d_model, max(n_subtypes, 1))
    def forward(self, x):
        h = self.proj(x.float()) + self.pos[:, :x.size(1)]
        h = self.encoder(h)
        q = self.query.expand(h.size(0), -1, -1)
        pooled, attn = self.attn_pool(q, h, h, need_weights=True)
        z = self.norm(pooled.squeeze(1))
        return (self.score_head(z).squeeze(-1), self.subtype_head(z.detach()),
                attn.squeeze(1).squeeze(1))

def window_starts(n, cfg):
    return np.array([0]) if n <= cfg.window_len else np.arange(0, n - cfg.window_len + 1,
                                                               cfg.window_stride)

def get_window(feat, s, cfg):
    w = feat[s:s + cfg.window_len]
    if len(w) < cfg.window_len:
        w = np.concatenate([w, np.repeat(w[-1:], cfg.window_len - len(w), axis=0)])
    return w

def enable_mc_dropout(m):
    for mod in m.modules():
        if isinstance(mod, nn.Dropout):
            mod.train()

@torch.no_grad()
def window_scores_mc(model, feat, cfg, temperature, chunk=512):
    starts = window_starts(len(feat), cfg)
    ps, ss = [], []
    for s in range(0, len(starts), chunk):
        w = torch.from_numpy(np.stack([get_window(feat, st, cfg)
                                       for st in starts[s:s + chunk]])).to(DEVICE)
        runs = []
        model.eval(); enable_mc_dropout(model)
        for _ in range(cfg.mc_passes):
            runs.append(torch.sigmoid(model(w)[0].float() / temperature).cpu().numpy())
        model.eval()
        runs = np.stack(runs)
        ps.append(runs.mean(0)); ss.append(runs.std(0))
    return np.concatenate(ps), np.concatenate(ss), starts

class AlarmSuppressor:
    def __init__(self, cfg, on):
        self.cfg, self.on, self.off = cfg, on, on * 0.6
        self.reset()
    def reset(self):
        self.ema, self.alarm, self.votes = 0.0, False, deque(maxlen=7)
    def update(self, p, unc=0.0):
        self.ema = 0.4 * p + 0.6 * self.ema
        q = self.ema >= self.on and unc <= 0.15
        self.votes.append(int(q))
        pers = sum(self.votes) / 7
        if not self.alarm and q and sum(self.votes) >= 4:
            self.alarm = True
        elif self.alarm and self.ema < self.off:
            self.alarm = False
        return {"qualified": q, "persistence": pers, "alarm": self.alarm}

def run_gates(p_w, s_w, cfg, on):
    sup = AlarmSuppressor(cfg, on)
    tr = [sup.update(p, u) for p, u in zip(p_w, s_w)]
    return {"candidate": any(t["alarm"] for t in tr),
            "persistence": round(max(t["persistence"] for t in tr), 3)}

class FaceAnonymizer:
    def __init__(self, blocks=6):
        self.blocks, self.det = blocks, None
        try:
            self.det = cv2.CascadeClassifier(os.path.join(
                cv2.data.haarcascades, "haarcascade_frontalface_default.xml"))
        except AttributeError:
            pass
    def anonymize(self, fr):
        if self.det is None:
            return fr.copy(), 0
        faces = self.det.detectMultiScale(cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY),
                                          1.1, 4, minSize=(14, 14))
        out = fr.copy()
        for (x, y, w, h) in faces:
            roi = out[y:y + h, x:x + w]
            small = cv2.resize(roi, (self.blocks, self.blocks))
            out[y:y + h, x:x + w] = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
        return out, len(faces)

anonymizer = FaceAnonymizer()

def decode_video(path, target_fps=4.0, max_seconds=120):
    """Decode an uploaded clip at ~target_fps, capped at max_seconds of footage."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return []
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0) or 30.0
    step = max(1, round(fps / target_fps))
    max_src = int(max_seconds * fps)
    frames, i = [], 0  
    while cap.grab() and i < max_src:
        if i % step == 0:
            ok, fr = cap.retrieve()
            if ok:
                frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        i += 1
    cap.release()
    return frames

print("Core AEGIS components ready.")

Core AEGIS components ready.


In [4]:
# ================= model bring-up: load trained artifacts OR quick-train fallback ========
def quick_discover():
    """Compact dataset discovery for samples / quick-train (frame images or .mp4)."""
    recs = []
    root = Path("/kaggle/input")
    if not root.is_dir():
        return pd.DataFrame(recs)
    for ds in sorted(root.iterdir()):
        for dirpath, _dirs, files in os.walk(ds):
            leaf = Path(dirpath)
            imgs = sorted(f for f in files if os.path.splitext(f)[1].lower()
                          in (".png", ".jpg", ".jpeg"))
            vids = [f for f in files if os.path.splitext(f)[1].lower()
                    in (".mp4", ".avi", ".mov", ".mkv")]
            cls = leaf.name if leaf.parent.name.lower() in ("train", "test") else leaf.parent.name
            groups = defaultdict(list)
            for f in imgs:
                m = FRAME_SUFFIX_RE.match(os.path.splitext(f)[0])
                groups[m.group(1) if m else "seq"].append(
                    (int(m.group(2)) if m else 0, str(leaf / f)))
            for g, fr in groups.items():
                if len(fr) >= 32:
                    recs.append(dict(cls=cls, group=g, kind="frames",
                                     paths=[p for _, p in sorted(fr)]))
            for v in vids:
                stem = os.path.splitext(v)[0]
                recs.append(dict(cls=parse_xd_label(stem) or cls, group=stem, kind="video",
                                 paths=[str(leaf / v)]))
        if recs:
            break
    df = pd.DataFrame(recs)
    if len(df):
        df["is_anom"] = ~df.cls.str.lower().str.contains("|".join(NORMAL_KEYWORDS))
    return df

DATA = quick_discover()
print(f"Dataset discovery: {len(DATA)} sequences"
      + (f" ({int(DATA.is_anom.sum())} anomalous / {int((~DATA.is_anom).sum())} normal)"
         if len(DATA) else " — no dataset attached (upload-only mode)"))

def frames_of(rec, cap=400):
    if rec["kind"] == "video":
        return decode_video(rec["paths"][0], 4.0, cap / 4.0)
    idx = np.unique(np.linspace(0, len(rec["paths"]) - 1,
                                min(cap, len(rec["paths"]))).round().astype(int))
    out = []
    for i in idx:
        im = cv2.imread(rec["paths"][int(i)], cv2.IMREAD_COLOR)
        if im is not None:
            out.append(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    return out

if ART is not None:
    CFGD = SimpleNamespace(**{k: ART[k] for k in
                              ("window_len", "window_stride", "d_model", "n_heads",
                               "n_layers", "dropout", "mc_passes")})
    ANOMALY_CLASSES = ART["anomaly_classes"]
    backbone, FEAT_DIM, BB_SPEC = build_backbone(ART["backbone"])
    model = SafetyFormer(ART["feat_dim"], CFGD, len(ANOMALY_CLASSES)).to(DEVICE)
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    model.eval()
    TEMPERATURE, ALARM_THRESHOLD = ART["temperature"], ART["alarm_threshold"]
    MIL_ALPHA = ART["mil_alpha"]
    print(f"Trained SafetyFormer restored ({ART['backbone_tag']} backbone, "
          f"alpha={MIL_ALPHA}, thr={ALARM_THRESHOLD:.3f}, T={TEMPERATURE:.3f}).")
else:
    # ---------- QUICK-TRAIN FALLBACK (clearly reduced scale; ~12 min on a T4) ----------
    assert len(DATA), "No dataset attached and no artifacts found — attach one of them."
    print("QUICK-TRAIN fallback engaged (subset model — attach the training notebook's "
          "output for the full-quality model).")
    CFGD = SimpleNamespace(window_len=16, window_stride=8, d_model=256, n_heads=4,
                           n_layers=2, dropout=0.2, mc_passes=8)
    MIL_ALPHA = 0.15
    backbone, FEAT_DIM, BB_SPEC = build_backbone("vit_base_patch32_clip_224.openai")
    sub = (DATA.groupby("cls", group_keys=False)
               .apply(lambda g: g.sample(min(len(g), 24), random_state=42))
               .reset_index(drop=True))
    va = sub.sample(frac=0.2, random_state=42)
    tr = sub.drop(va.index)
    ANOMALY_CLASSES = sorted(sub[sub.is_anom].cls.unique())
    FE = {}
    for _, r in sub.iterrows():
        fr = frames_of(r.to_dict(), cap=240)
        if len(fr) >= 8:
            FE[r.group] = embed_frames(fr, backbone, BB_SPEC)
    tr, va = tr[tr.group.isin(FE)], va[va.group.isin(FE)]
    print(f"  embedded {len(FE)} sequences ({len(tr)} train / {len(va)} val)")
    model = SafetyFormer(FEAT_DIM, CFGD, len(ANOMALY_CLASSES)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    S2I = {c: i for i, c in enumerate(ANOMALY_CLASSES)}
    for ep in range(8):
        model.train()
        rows = tr.sample(frac=1.0, random_state=ep)
        losses = []
        for _, r in rows.iterrows():
            feat = FE[r.group]
            starts = window_starts(len(feat), CFGD)
            if len(starts) > 48:
                starts = np.random.choice(starts, 48, replace=False)
            w = torch.from_numpy(np.stack([get_window(feat, s, CFGD)
                                           for s in starts])).to(DEVICE)
            wl, st, _ = model(w)
            k = max(1, math.ceil(MIL_ALPHA * len(wl)))
            v = wl.topk(k).values.mean()
            loss = F.binary_cross_entropy_with_logits(
                v.unsqueeze(0), torch.tensor([float(r.is_anom)], device=DEVICE))
            if r.is_anom and len(ANOMALY_CLASSES) > 1:
                loss = loss + 0.3 * F.cross_entropy(
                    st[wl.argmax()].unsqueeze(0),
                    torch.tensor([S2I[r.cls]], device=DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        print(f"  quick-train epoch {ep+1}/8 loss={np.mean(losses):.4f}")
    model.eval()
    vl, vy = [], []
    with torch.no_grad():
        for _, r in va.iterrows():
            feat = FE[r.group]
            starts = window_starts(len(feat), CFGD)
            w = torch.from_numpy(np.stack([get_window(feat, s, CFGD)
                                           for s in starts])).to(DEVICE)
            wl = model(w)[0]
            k = max(1, math.ceil(MIL_ALPHA * len(wl)))
            vl.append(wl.topk(k).values.mean().item()); vy.append(float(r.is_anom))
    vl, vy = np.array(vl), np.array(vy)
    ts = torch.zeros(1, requires_grad=True)
    lb = torch.optim.LBFGS([ts], lr=0.05, max_iter=60)
    def _cl():
        lb.zero_grad()
        l = F.binary_cross_entropy_with_logits(
            torch.from_numpy(vl).float() / torch.exp(ts), torch.from_numpy(vy).float())
        l.backward(); return l
    lb.step(_cl)
    TEMPERATURE = float(torch.exp(ts))
    p_cal = 1 / (1 + np.exp(-vl / TEMPERATURE))
    from sklearn.metrics import precision_recall_curve, roc_auc_score
    pr, rc, th = precision_recall_curve(vy, p_cal)
    ok = rc[:-1] >= 0.85
    j = int(np.argmax(np.where(ok, pr[:-1], -1))) if ok.any() else int(
        np.argmax(2 * pr[:-1] * rc[:-1] / np.clip(pr[:-1] + rc[:-1], 1e-9, None)))
    ALARM_THRESHOLD = float(th[j])
    print(f"  quick model: val ROC-AUC={roc_auc_score(vy, p_cal):.3f} "
          f"thr={ALARM_THRESHOLD:.3f} T={TEMPERATURE:.3f} (subset-scale, demo only)")

Dataset discovery: 1865 sequences (930 anomalous / 935 normal)
QUICK-TRAIN fallback engaged (subset model — attach the training notebook's output for the full-quality model).


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

  embedded 336 sequences (269 train / 67 val)
  quick-train epoch 1/8 loss=1.0524
  quick-train epoch 2/8 loss=0.9933
  quick-train epoch 3/8 loss=0.9629
  quick-train epoch 4/8 loss=0.9166
  quick-train epoch 5/8 loss=0.9058
  quick-train epoch 6/8 loss=0.8460
  quick-train epoch 7/8 loss=0.8864
  quick-train epoch 8/8 loss=0.8683
  quick model: val ROC-AUC=0.937 thr=0.852 T=1.335 (subset-scale, demo only)


In [5]:
# ================= LOCAL multimodal Gemma 4 (identical to the training notebook) =========
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
try:
    from kaggle_secrets import UserSecretsClient
    _t = UserSecretsClient().get_secret("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception:
    pass

gemma_model, gemma_processor, GEMMA_MODEL_ID, LOCAL_GEMMA = None, None, None, False
if torch.cuda.is_available():
    _dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=_dt)
    for mid in ("google/gemma-4-E4B-it", "google/gemma-4-E2B-it"):
        try:
            print(f"Loading {mid} (4-bit NF4, local weights)...")
            gemma_processor = AutoProcessor.from_pretrained(mid, padding_side="left")
            gemma_model = AutoModelForImageTextToText.from_pretrained(
                mid, quantization_config=qc, device_map="auto", attn_implementation="sdpa")
            gemma_model.eval()
            GEMMA_MODEL_ID, LOCAL_GEMMA = mid, True
            break
        except Exception as e:
            print(f"  {mid} failed: {type(e).__name__}: {str(e)[:160]}")
if LOCAL_GEMMA:
    print(f"LOCAL GEMMA 4 ONLINE: {GEMMA_MODEL_ID} | "
          f"{gemma_model.get_memory_footprint()/1e9:.2f} GB | no inference API")
else:
    print("Gemma 4 unavailable -> app runs with the rule-based fallback, every output "
          "explicitly labeled as degraded (NOT Gemma).")

Loading google/gemma-4-E4B-it (4-bit NF4, local weights)...


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

LOCAL GEMMA 4 ONLINE: google/gemma-4-E4B-it | 9.23 GB | no inference API


In [6]:
# ================= verifier + fusion + evidence (identical logic) ========================
GEMMA_SYSTEM_PROMPT = (
    "You are the multimodal forensic verification stage of an edge public-safety system.\n"
    "A lightweight anomaly detector (SafetyFormer) has flagged a temporal window as a "
    "possible incident. You will see its telemetry, but treat it as an UNTRUSTED TRIAGE "
    "HYPOTHESIS: do not assume it is correct.\n"
    "Independently inspect the anonymized evidence frames (chronological order, faces "
    "pixelated for privacy). Use only visually supported evidence. Carefully distinguish "
    "genuine safety incidents from ordinary activity, crowds, fast but benign motion, "
    "camera artifacts, and ambiguous scenes. Low image quality alone is not evidence of "
    "an incident.\n"
    "Respond with ONE valid JSON object and nothing else, using exactly these keys:\n"
    '{"incident_supported": true|false, "incident_type": "<short label or none>", '
    '"severity": <int 1-5>, "visual_evidence": ["<observation>", ...], '
    '"false_positive_likelihood": <float 0-1>, "confidence": <float 0-1>, '
    '"recommended_action": "none"|"log_only"|"notify_operator"|"review_queue"|"urgent_dispatch"}'
)
ALLOWED_ACTIONS = ("none", "log_only", "notify_operator", "review_queue", "urgent_dispatch")

def extract_json(text):
    text = re.sub(r"```(?:json)?", " ", text)
    start = text.find("{")
    while start != -1:
        depth, in_str, esc = 0, False, False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                if esc: esc = False
                elif ch == "\\": esc = True
                elif ch == '"': in_str = False
            elif ch == '"': in_str = True
            elif ch == "{": depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i + 1])
                    except json.JSONDecodeError:
                        break
        start = text.find("{", start + 1)
    return None

def validate_verdict(v):
    if not isinstance(v, dict):
        return None
    try:
        ev = v.get("visual_evidence", [])
        if isinstance(ev, str):
            ev = [ev]
        act = str(v.get("recommended_action", "review_queue")).lower().strip()
        return {"incident_supported": bool(v["incident_supported"]),
                "incident_type": str(v.get("incident_type", "unknown"))[:60],
                "severity": int(min(5, max(1, int(float(v.get("severity", 1)))))),
                "visual_evidence": [str(e)[:200] for e in ev][:8],
                "false_positive_likelihood": float(min(1, max(0, float(v["false_positive_likelihood"])))),
                "confidence": float(min(1, max(0, float(v["confidence"])))),
                "recommended_action": act if act in ALLOWED_ACTIONS else "review_queue"}
    except (KeyError, TypeError, ValueError):
        return None

def gemma_verify(pil_frames, telemetry):
    if not LOCAL_GEMMA:
        m = re.search(r"anomaly score: ([\d.]+)", telemetry)
        p = float(m.group(1)) if m else 0.5
        return ({"incident_supported": p >= 0.75, "incident_type": "unverified",
                 "severity": 3 if p >= 0.75 else 1,
                 "visual_evidence": ["FALLBACK: no visual inspection performed"],
                 "false_positive_likelihood": round(1 - p, 2), "confidence": 0.30,
                 "recommended_action": "review_queue"},
                {"verifier": "rule-based fallback (degraded — NOT Gemma)", "latency_s": 0.0})
    text = telemetry
    for attempt in range(2):
        messages = [{"role": "system", "content": [{"type": "text", "text": GEMMA_SYSTEM_PROMPT}]},
                    {"role": "user", "content":
                        [{"type": "image", "image": im} for im in pil_frames]
                        + [{"type": "text", "text": text}]}]
        inputs = gemma_processor.apply_chat_template(
            messages, tokenize=True, return_dict=True, return_tensors="pt",
            add_generation_prompt=True).to(gemma_model.device)
        n_in = inputs["input_ids"].shape[-1]
        t0 = time.perf_counter()
        with torch.inference_mode():
            out = gemma_model.generate(**inputs, max_new_tokens=512, do_sample=False)
        dt = time.perf_counter() - t0
        raw = gemma_processor.decode(out[0][n_in:], skip_special_tokens=True)
        torch.cuda.empty_cache()
        v = validate_verdict(extract_json(raw))
        if v is not None:
            return v, {"verifier": f"{GEMMA_MODEL_ID} (LOCAL, 4-bit NF4)",
                       "latency_s": round(dt, 1), "raw": raw,
                       "tok_per_s": round((out.shape[-1] - n_in) / max(dt, 1e-6), 1)}
        text = telemetry + "\nYour previous reply was not valid JSON. Return ONLY the JSON object."
    return ({"incident_supported": False, "incident_type": "parse_failure", "severity": 1,
             "visual_evidence": ["Gemma output unparseable twice -> conservative review"],
             "false_positive_likelihood": 0.5, "confidence": 0.2,
             "recommended_action": "review_queue"},
            {"verifier": f"{GEMMA_MODEL_ID} (output unparseable — routed to review)",
             "latency_s": 0.0})

def decide(p_cal, persistence, verdict, thr):
    g = verdict
    if (g["incident_supported"] and g["confidence"] >= 0.60
            and g["false_positive_likelihood"] <= 0.40 and p_cal >= thr):
        return "ESCALATE", "detector above threshold AND Gemma visually confirms"
    if not g["incident_supported"] and p_cal >= 0.92 and persistence >= 0.60:
        return "REVIEW", "Gemma rejects, but temporal evidence is extreme — routed to a human"
    if not g["incident_supported"] and g["false_positive_likelihood"] >= 0.60:
        return "SUPPRESS", "Gemma finds no visual support — likely false alarm"
    if g["incident_supported"]:
        return "REVIEW", "Gemma sees a possible incident but with reservations"
    return "REVIEW", "ambiguous evidence — conservative human review"

def select_evidence_frames(frames_anon, p_win, starts, cfg, n=6, out_size=432):
    peak = int(np.argmax(p_win))
    lo = hi = peak
    floor = max(0.6 * p_win[peak], 0.05)
    while lo - 1 >= 0 and p_win[lo - 1] >= floor:
        lo -= 1
    while hi + 1 < len(p_win) and p_win[hi + 1] >= floor:
        hi += 1
    f0 = int(starts[lo])
    f1 = int(min(starts[hi] + cfg.window_len, len(frames_anon)))
    cand = np.unique(np.linspace(f0, f1 - 1, n * 3).round().astype(int))
    kept, last = [], None
    for i in cand:
        g = cv2.cvtColor(frames_anon[i], cv2.COLOR_RGB2GRAY).astype(np.float32)
        if last is None or np.abs(g - last).mean() > 3.0:
            kept.append(int(i)); last = g
    if len(kept) < 2:
        kept = list(np.unique(np.linspace(f0, f1 - 1, n).round().astype(int)))
    sel = np.unique(np.linspace(0, len(kept) - 1, min(n, len(kept))).round().astype(int))
    idx = [kept[i] for i in sel]
    imgs = [cv2.resize(frames_anon[i], (out_size, out_size),
                       interpolation=cv2.INTER_CUBIC) for i in idx]
    return imgs, idx, (lo, hi)

print("Verifier, fusion and evidence selection ready.")

Verifier, fusion and evidence selection ready.


In [7]:
# ================= backend: one function = the whole pipeline (staged generator) =========
def timeline_figure(p_win, thr, region=None):
    fig, ax = plt.subplots(figsize=(8.6, 2.6), dpi=110)
    ax.plot(p_win, color="#0f766e", lw=1.8)
    ax.axhline(thr, color="#dc2626", ls="--", lw=1)
    if region:
        ax.axvspan(region[0], region[1], color="#dc2626", alpha=0.12)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("temporal window (~4 s each)", fontsize=8)
    ax.set_ylabel("P(anomalous)", fontsize=8)
    ax.tick_params(labelsize=7)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    fig.tight_layout()
    return fig

def analyze_stream(video_path):
    """Generator: yields (status, timeline, gallery, verdict_html, decision_html, stats_html)
    stage by stage so the UI reveals the pipeline as it happens."""
    def st(msg, step):
        steps = ["Decode", "Anonymize", "Triage", "Evidence", "Gemma 4", "Decision"]
        chips = "".join(
            f'<span class="chip {"done" if i < step else ("act" if i == step else "")}">{s}</span>'
            for i, s in enumerate(steps))
        return f'<div class="stepper">{chips}</div><div class="status">{msg}</div>'
    empty = gr.update()
    if not video_path:
        yield st("Upload a clip (or load a sample) and press Analyze.", -1), empty, empty, empty, empty, empty
        return
    T = {}
    t0 = time.perf_counter()
    yield st("Decoding video at 4 fps…", 0), None, None, "", "", ""
    frames = decode_video(video_path)
    T["decode"] = time.perf_counter() - t0
    if len(frames) < 8:
        yield st("Could not decode enough frames from this file.", 0), None, None, "", "", ""
        return
    t0 = time.perf_counter()
    yield st(f"{len(frames)} frames — pixelating faces on-device…", 1), None, None, "", "", ""
    _pairs = [anonymizer.anonymize(f) for f in frames]
    anon = [p[0] for p in _pairs]
    n_faces = sum(p[1] for p in _pairs)
    T["anonymize"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    yield st("SafetyFormer triage over dense temporal windows…", 2), None, None, "", "", ""
    feat = embed_frames(anon, backbone, BB_SPEC)
    p_win, s_win, starts = window_scores_mc(model, feat, CFGD, TEMPERATURE)
    k = max(1, math.ceil(MIL_ALPHA * len(p_win)))
    p_cal = float(np.sort(p_win)[-k:].mean())
    gates = run_gates(p_win, s_win, CFGD, ALARM_THRESHOLD)
    with torch.no_grad():
        _, stl, _ = model(torch.from_numpy(
            get_window(feat, int(starts[int(np.argmax(p_win))]), CFGD)).unsqueeze(0).to(DEVICE))
    subtype = ANOMALY_CLASSES[int(stl[0].argmax())] if ANOMALY_CLASSES else "unknown"
    T["triage"] = time.perf_counter() - t0
    fig = timeline_figure(p_win, ALARM_THRESHOLD)
    if not gates["candidate"] and p_cal < ALARM_THRESHOLD:
        d_html = ('<div class="banner suppress">NO INCIDENT — anomaly score '
                  f'{p_cal:.2f} below threshold {ALARM_THRESHOLD:.2f}; Gemma 4 was not '
                  'woken (selective compute)</div>')
        stats = stats_html(T, None, p_cal, gates, subtype, n_faces, len(p_win), gemma_used=False)
        yield st("Quiet clip — no candidate incident.", 5), fig, [], "", d_html, stats
        return
    t0 = time.perf_counter()
    yield st("Candidate incident — selecting anonymized evidence keyframes…", 3), fig, None, "", "", ""
    ev_imgs, ev_idx, region = select_evidence_frames(anon, p_win, starts, CFGD)
    fig = timeline_figure(p_win, ALARM_THRESHOLD, region)
    gallery = [(im, f"frame {i}") for im, i in zip(ev_imgs, ev_idx)]
    T["evidence"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    yield st("Gemma 4 is inspecting the evidence locally…", 4), fig, gallery, "", "", ""
    telemetry = ("UNTRUSTED TRIAGE HYPOTHESIS from the lightweight detector — verify "
                 "independently:\n"
                 f"- calibrated anomaly score: {p_cal:.2f} (operating threshold {ALARM_THRESHOLD:.2f})\n"
                 f"- top incident-type hypothesis: {subtype}\n"
                 f"- detector epistemic uncertainty (MC-dropout std): {float(s_win.max()):.3f}\n"
                 f"- temporal persistence: {gates['persistence']:.2f}\n"
                 f"- the {len(ev_imgs)} frames span the highest-anomaly temporal region, "
                 "chronological, faces pixelated on-device.\n"
                 "Inspect the frames and return ONLY the JSON object.")
    verdict, meta = gemma_verify([Image.fromarray(im) for im in ev_imgs], telemetry)
    T["gemma"] = time.perf_counter() - t0
    decision, why = decide(p_cal, gates["persistence"], verdict, ALARM_THRESHOLD)
    v_html = verdict_html(verdict, meta)
    d_html = (f'<div class="banner {decision.lower()}">{decision}'
              f'<span class="why">{why}</span></div>')
    stats = stats_html(T, meta, p_cal, gates, subtype, n_faces, len(p_win), gemma_used=True)
    yield st("Done.", 5), fig, gallery, v_html, d_html, stats

def verdict_html(v, meta):
    ev = "".join(f"<li>{e}</li>" for e in v["visual_evidence"])
    bar = lambda x: (f'<div class="bar"><div class="fill" '
                     f'style="width:{x*100:.0f}%"></div></div>')
    return f"""
    <div class="card fade">
      <div class="cardtitle">GEMMA 4 VISUAL VERIFICATION
        <span class="tag">{meta['verifier']}</span></div>
      <div class="grid">
        <div><b>Incident supported</b><br>{'YES' if v['incident_supported'] else 'NO'}</div>
        <div><b>Category</b><br>{v['incident_type']}</div>
        <div><b>Severity</b><br>{'●' * v['severity']}{'○' * (5 - v['severity'])}</div>
        <div><b>Recommended action</b><br>{v['recommended_action']}</div>
      </div>
      <div class="meters">
        <div>Confidence {v['confidence']:.2f}{bar(v['confidence'])}</div>
        <div>False-positive likelihood {v['false_positive_likelihood']:.2f}{bar(v['false_positive_likelihood'])}</div>
      </div>
      <div class="evtitle">Visual evidence cited</div><ul>{ev}</ul>
    </div>"""

def stats_html(T, meta, p_cal, gates, subtype, n_faces, n_win, gemma_used):
    lat = " · ".join(f"{k} {v:.1f}s" for k, v in T.items())
    g = (f"Gemma latency {meta['latency_s']}s ({meta.get('tok_per_s', '—')} tok/s) · "
         if (gemma_used and meta) else "Gemma not activated · ")
    return (f'<div class="stats fade">SafetyFormer score {p_cal:.2f} · '
            f'hypothesis {subtype} · persistence {gates["persistence"]:.2f} · '
            f'{n_win} windows analyzed · {n_faces} face regions pixelated · {g}'
            f'stage latency: {lat} · all inference LOCAL — no cloud API</div>')

print("Backend analyze function ready.")

Backend analyze function ready.


In [8]:
# ================= sample CCTV clips (rendered from the attached dataset) ================
SAMPLES = {}
def make_sample(rec, name, seconds=45):
    fr = frames_of(rec, cap=int(seconds * 4))
    if len(fr) < 16:
        return None
    raw = str(WORK / f"raw_{name}.mp4")
    out = str(WORK / f"sample_{name}.mp4")
    h, w = 256, 256
    vw = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*"mp4v"), 4.0, (w, h))
    for f in fr:
        vw.write(cv2.cvtColor(cv2.resize(f, (w, h)), cv2.COLOR_RGB2BGR))
    vw.release()
    if shutil.which("ffmpeg"):   # h264 so it previews in the browser
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw,
                        "-vcodec", "libx264", "-pix_fmt", "yuv420p", out])
        os.remove(raw)
        return out
    return raw

if len(DATA):
    rng = np.random.default_rng(7)
    anoms = DATA[DATA.is_anom]
    norms = DATA[~DATA.is_anom]
    picks = []
    if len(anoms):
        for _, r in anoms.sample(min(2, len(anoms)), random_state=7).iterrows():
            picks.append((f"{r.cls} (anomalous sample)", r.to_dict()))
    if len(norms):
        for _, r in norms.sample(min(2, len(norms)), random_state=7).iterrows():
            picks.append((f"{r.cls} (normal sample)", r.to_dict()))
    for label, rec in picks:
        p = make_sample(rec, re.sub(r"\W+", "_", label))
        if p:
            SAMPLES[label] = p
print(f"Sample clips ready: {list(SAMPLES) or 'none (upload-only mode)'}")

Sample clips ready: ['Vandalism (anomalous sample)', 'Shoplifting (anomalous sample)', 'NormalVideos (normal sample)']


In [10]:
# ================= frontend: minimal, clean, animated =====================================
CSS = """
:root { --ink:#111827; --sub:#6b7280; --teal:#0f766e; --bg:#fafaf9; --line:#e7e5e4; }
.gradio-container { background: var(--bg) !important; max-width: 1180px !important;
  margin: auto !important; font-family: -apple-system, 'Segoe UI', Roboto, sans-serif; }
#hdr h1 { font-weight: 650; letter-spacing: -.02em; margin-bottom: 0; color: var(--ink); }
#hdr p { color: var(--sub); margin-top: 4px; }
.stepper { display:flex; gap:8px; margin: 6px 0 10px; }
.chip { padding: 4px 12px; border-radius: 999px; font-size: 12px; color: var(--sub);
  background:#f1f0ef; border:1px solid var(--line); transition: all .35s ease; }
.chip.act { background: var(--teal); color:#fff; border-color: var(--teal);
  box-shadow: 0 2px 10px rgba(15,118,110,.35); }
.chip.done { background:#d1fae5; color:#065f46; border-color:#a7f3d0; }
.status { color: var(--sub); font-size: 13px; min-height: 18px; }
.card { background:#fff; border:1px solid var(--line); border-radius:14px; padding:16px 18px;
  box-shadow: 0 1px 4px rgba(0,0,0,.04); }
.cardtitle { font-size:12px; letter-spacing:.08em; color:var(--sub); margin-bottom:10px; }
.tag { float:right; font-size:10px; background:#ecfdf5; color:#065f46; padding:2px 8px;
  border-radius:999px; }
.grid { display:grid; grid-template-columns: repeat(4,1fr); gap:10px; font-size:13px;
  color:var(--ink); }
.meters { display:grid; grid-template-columns:1fr 1fr; gap:14px; margin-top:12px;
  font-size:12px; color:var(--sub); }
.bar { height:6px; background:#f1f0ef; border-radius:999px; margin-top:4px; overflow:hidden; }
.fill { height:100%; background:var(--teal); border-radius:999px;
  transition: width .8s cubic-bezier(.2,.8,.2,1); }
.evtitle { font-size:12px; color:var(--sub); margin-top:12px; }
.card ul { margin:6px 0 0 18px; font-size:13px; color:var(--ink); }
.banner { border-radius:14px; padding:18px 22px; font-size:22px; font-weight:700;
  letter-spacing:.02em; color:#fff; animation: pop .5s cubic-bezier(.2,.8,.2,1); }
.banner .why { display:block; font-size:13px; font-weight:400; opacity:.92; margin-top:4px; }
.banner.escalate { background:linear-gradient(120deg,#dc2626,#b91c1c); }
.banner.review   { background:linear-gradient(120deg,#d97706,#b45309); }
.banner.suppress { background:linear-gradient(120deg,#4b5563,#374151); font-size:16px; }
.stats { font-size:11.5px; color:var(--sub); padding: 8px 2px; }
.fade { animation: fadein .6s ease; }
@keyframes fadein { from {opacity:0; transform:translateY(6px);} to {opacity:1;} }
@keyframes pop { from {opacity:0; transform:scale(.97);} to {opacity:1; transform:scale(1);} }
footer { display:none !important; }
"""

with gr.Blocks(css=CSS, title="AEGIS — Gemma 4 Public Safety", theme=gr.themes.Base()) as app:
    gr.HTML('<div id="hdr"><h1>🛡️ AEGIS</h1><p>Continuous cheap triage · anonymized '
            'evidence · <b>local Gemma 4</b> forensic verification — no video ever '
            'leaves this machine.</p></div>')
    with gr.Row():
        with gr.Column(scale=5):
            video_in = gr.Video(label="CCTV clip (mp4/avi — first 120 s analyzed)",
                                sources=["upload"], height=300)
            if SAMPLES:
                sample_dd = gr.Dropdown(list(SAMPLES), label="…or load a sample clip",
                                        value=None)
                sample_dd.change(lambda k: SAMPLES.get(k), sample_dd, video_in)
            go = gr.Button("Analyze", variant="primary", size="lg")
            gr.HTML('<div class="stats">Pipeline: decode → face pixelation → SafetyFormer '
                    'dense-window triage → suppression gates → evidence keyframes → '
                    'Gemma 4 (4-bit, on-GPU) → decision fusion</div>')
        with gr.Column(scale=7):
            status = gr.HTML()
            timeline = gr.Plot(label="Window-level anomaly timeline")
            gallery = gr.Gallery(label="Anonymized evidence — exactly what Gemma 4 inspected",
                                 columns=6, height=140, object_fit="cover")
            verdict_out = gr.HTML()
            decision_out = gr.HTML()
            stats_out = gr.HTML()
    go.click(analyze_stream, inputs=[video_in],
             outputs=[status, timeline, gallery, verdict_out, decision_out, stats_out])

app.queue().launch(share=True, prevent_thread_lock=True)
print("\n" + "=" * 70)
print("APP IS LIVE — use the PUBLIC 'gradio.live' URL above for judging.")
print("Keep this Kaggle session open for the duration of the demo.")
print("=" * 70)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://f0bfce696edd140188.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



APP IS LIVE — use the PUBLIC 'gradio.live' URL above for judging.
Keep this Kaggle session open for the duration of the demo.
